<h2><b>计算机高等教育通用教材</b></h2>
<h2>机器学习 Machine learning</h2>
<hr>
<h5>第二部分：无监督学习 unsupervised learning</h5>
<h5>第七章：聚类与空间划分 K-Means (K-平均值)</h5>
<hr>
<h3><b>实验七：基于 K-Means 的 NBA 球员技术特点聚类分析</b></h3>
<hr>
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html'>查看KMeans源代码(sklearn)</a><br>
<br>

> **适合人群** ：前面六章，我们学习的全是“监督学习”——数据里自带标准答案（标签 y），模型照着答案学。但现实中，大多数数据是没有标签的。比如给你一堆客户的消费记录，没有任何人提前告诉你他们属于哪类人。
> 本章，我们将进入“无监督学习”的世界。算法需要自己在混沌的数据中寻找相似性，实现物以类聚、人以群分。
<hr>

#### 第0步：测试python与虚拟环境

In [ ]:
print("Hello K-Means Clustering!")
import pip
print("Pip version:", pip.__version__)

<hr><hr>

#### 第一步：import库 & 导入数据
<hr>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [ ]:
# 教材中使用了 players.csv。为了保证代码的一键可运行性，
# 我们利用严格的统计学分布，模拟生成 286 位 NBA 球员的赛场表现数据。
# 字段包括：球员编号、得分、命中率、三分命中率、罚球命中率。
# 注意：在无监督学习中，我们没有任何关于“球员究竟属于什么角色”的标签！

np.random.seed(42)

# 我们在底层偷偷设定三种类型的球员数据，混在一起给模型去猜：
# 1. 核心得分手 (79人)：高得分，高命中率
pts_core = np.random.normal(26.5, 3.5, 79)
fg_core = np.random.normal(0.48, 0.04, 79)
fg3_core = np.random.normal(0.38, 0.05, 79)
ft_core = np.random.normal(0.85, 0.05, 79)

# 2. 组织/蓝领球员 (154人)：中等偏低得分，命中率尚可
pts_role = np.random.normal(12.0, 3.0, 154)
fg_role = np.random.normal(0.44, 0.05, 154)
fg3_role = np.random.normal(0.34, 0.06, 154)
ft_role = np.random.normal(0.76, 0.08, 154)

# 3. 边缘/新秀球员 (53人)：低得分，命中率极其不稳定
pts_bench = np.random.normal(4.5, 2.0, 53)
fg_bench = np.random.normal(0.38, 0.08, 53)
fg3_bench = np.random.normal(0.28, 0.10, 53)
ft_bench = np.random.normal(0.65, 0.12, 53)

# 拼接并打乱顺序
pts = np.concatenate([pts_core, pts_role, pts_bench])
fg = np.concatenate([fg_core, fg_role, fg_bench])
fg3 = np.concatenate([fg3_core, fg3_role, fg3_bench])
ft = np.concatenate([ft_core, ft_role, ft_bench])

# 限制极值，避免概率超过 1 或小于 0，得分不能小于 0
pts = np.clip(pts, 0, 40)
fg = np.clip(fg, 0.1, 0.7)
fg3 = np.clip(fg3, 0.0, 0.6)
ft = np.clip(ft, 0.2, 1.0)

player_ids = np.arange(1, 287)
np.random.shuffle(player_ids) # 彻底打乱

df = pd.DataFrame({
    '编号': player_ids,
    '得分': pts,
    '命中率': fg,
    '三分命中率': fg3,
    '罚球命中率': ft
})

print(f'数据集大小: {df.shape}')
df.head()

<hr><hr>

#### 第二步：数据观察与预处理（归一化）
<hr>

聚类算法的核心是“算距离”。既然要算距离，就绝对不能容忍不同量纲的存在。
比如“得分”是 20 多，“命中率”是 0.4 多。如果不处理，模型只会盯着得分看，完全忽视命中率。

In [ ]:
df.info()

In [ ]:
# 数据转换：使用 MinMaxScaler 将所有核心特征归一化至 [0, 1] 区间。
# 公式：X_scaled = (X - X_min) / (X_max - X_min)
# 我们不使用 StandardScaler，是因为各种命中率本身就是 [0, 1] 之间的比例，
# 用 MinMaxScaler 把得分也压扁到 [0, 1] 是最合理的处理方式。

X_raw = df[['得分', '命中率', '三分命中率', '罚球命中率']]

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

print("归一化后的数据矩阵前2行：")
print(X_scaled[:2])

<hr><hr>

#### 第三步：参数查找 —— 怎么确定分为几类（找最优的 K）？
<hr>

K-Means 算法最大的痛点就是：你需要事先告诉它分成 K 类。
但是我连数据长什么样都不知道，我怎么知道该分几类？
我们需要用两种统计学武器来帮我们决定：**肘部法则 (Elbow Method)** 和 **轮廓系数 (Silhouette Score)**。

In [ ]:
k_values = range(2, 11)  # 尝试把球员分成 2 到 10 类
inertias = []            # 存放每个 K 下的簇内误差平方和
silhouette_scores = []   # 存放每个 K 下的轮廓系数

for k in k_values:
    # init='k-means++' 极其重要！普通的 KMeans 是纯随机找初始点，容易陷入局部死胡同。
    # k-means++ 会尽量让初始化的几个中心点离得远一点，能极大提升聚类质量。
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    
    # 聚类并给每个样本贴上标签
    labels = kmeans.fit_predict(X_scaled)
    
    # 记录 Inertia（簇内距离和，越小越好，但越分得多必然越小）
    inertias.append(kmeans.inertia_)
    
    # 记录 轮廓系数（判断分得合不合理，取值 [-1, 1]，越大越好）
    sil_score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(sil_score)

In [ ]:
# 配置中文字体
import matplotlib.font_manager as fm
zh_fonts = [f.name for f in fm.fontManager.ttflist 
            if any(kw in f.name for kw in ['Hei', 'Song', 'CJK', 'Chinese', 'SC', 'TC', 'Gothic', 'SimHei'])]
if zh_fonts:
    plt.rcParams['font.family'] = zh_fonts[0]
plt.rcParams['axes.unicode_minus'] = False 

# 画出两种指标的折线图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 图1：肘部法则 (Inertia)
ax1.plot(k_values, inertias, marker='o', color='steelblue')
ax1.set_title('肘部法则 (Inertia) 寻找最优 K')
ax1.set_xlabel('聚类簇数 K')
ax1.set_ylabel('簇内误差平方和 (越小越拥挤)')
ax1.grid(alpha=0.3)

# 图2：轮廓系数 (Silhouette Score)
ax2.plot(k_values, silhouette_scores, marker='o', color='darkorange')
ax2.set_title('轮廓系数 (Silhouette) 评估聚类性能')
ax2.set_xlabel('聚类簇数 K')
ax2.set_ylabel('轮廓系数 (越大越清晰)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 结果分析：
# 1. 在肘部法则图里，K=3 的时候正好处于下降趋势明显放缓的“手肘”位置。
# 2. 在轮廓系数图里，K=3 拿到了最高分。
# 这说明，基于现有的 4 项数据指标，把这批 NBA 球员分为 3 类是最科学的。

<hr><hr>

#### 第四步：聚类可视化 与 簇分析（为数据赋予意义）
<hr>

In [ ]:
# 确定了最优 K=3，我们正式训练最终的 KMeans 模型
best_k = 3
final_kmeans = KMeans(n_clusters=best_k, init='k-means++', random_state=42, n_init=10)
df['聚类结果'] = final_kmeans.fit_predict(X_scaled)

# 看看每个簇的聚类中心是什么样子（我们将归一化的中心点，反向还原成真实的得分和命中率）
cluster_centers_scaled = final_kmeans.cluster_centers_
cluster_centers_real = scaler.inverse_transform(cluster_centers_scaled)

centers_df = pd.DataFrame(cluster_centers_real, columns=X_raw.columns)
centers_df.index.name = '簇编号'
print("【各簇的聚类中心（典型球员画像）】")
print(centers_df)

In [ ]:
# 分析各簇的样本数量，结合画像，给各个簇强行赋予人类能懂的业务意义
cluster_counts = df['聚类结果'].value_counts().sort_index()

print("\n【簇分析与实际类别映射】")
for cluster_id in range(best_k):
    count = cluster_counts[cluster_id]
    pts = centers_df.loc[cluster_id, '得分']
    fg = centers_df.loc[cluster_id, '命中率']
    
    print(f"簇 {cluster_id} (共 {count} 人):")
    if pts > 20:
        print("  -> 画像：得分极高，命中率优秀。")
        print("  -> 映射：球队的【核心进攻球员/球星】。")
    elif pts > 10:
        print("  -> 画像：得分中等，命中率稳定。")
        print("  -> 映射：得分能力一般的【组织型球员/首发蓝领】。")
    else:
        print("  -> 画像：得分极低，各项命中率惨淡。")
        print("  -> 映射：上场时间极少的【饮水机管理员/新秀球员】。")

In [ ]:
# 可视化展示：因为我们有 4 个特征，四维空间画不出来。
# 我们必须用 PCA（主成分分析）把 4 维压成 2 维，才能在屏幕上画出散点图。

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

plt.figure(figsize=(8, 6))
colors = ['red', 'green', 'blue']
labels = ['核心球星', '组织蓝领', '边缘新秀']

for i in range(best_k):
    cluster_data = df[df['聚类结果'] == i]
    plt.scatter(cluster_data['PC1'], cluster_data['PC2'], 
                c=colors[i], label=f'簇 {i} ({labels[i]})', alpha=0.6)

plt.title('NBA 球员 KMeans 聚类分布 (PCA降维二维展示)')
plt.xlabel('主成分 1')
plt.ylabel('主成分 2')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 从图中可以看出，三类球员在经过 PCA 降维后，依然保持着非常清晰的边界。

<hr><hr>

#### 第五步：【拓展提高】再探维度诅咒（Curse of Dimensionality）
<hr>

在第一章讲解 KNN 的时候，我们第一次提到了“维度诅咒”。

**什么是维度诅咒？**
在刚才的聚类里，我们只用了 4 个特征（维度）。如果我给你 1000 个特征呢？包括球员的身高、体重、臂展、视力、肺活量、玩某款游戏的段位、家里宠物的数量……

按照人类的直觉，信息越多，聚类应该越精准，对吧？
错！大错特错！

K-Means 和 KNN 的底层逻辑，都是在空间中算**欧式距离（直线距离）**。

当维度急剧飙升到几百上千维时，数学上会出现一个极其反直觉的恐怖现象：
**高维空间大得超乎想象。在几千维的空间里，任何一个点到其他所有点的距离，都会变得几乎一模一样。**

换句话说，詹姆斯（球星）到库里（球星）的距离，跟詹姆斯到一个刚选秀的菜鸟的距离，在 1000 维空间里算出来，居然是差不多的。

因为每个人的 1000 个属性里，总有几个特别强，总有几个特别弱。当 1000 个平方差加起来再开根号时，巨大的差异全被平均掉了。
距离失去了区分度，K-Means 瞬间变成瞎子，根本分不出谁和谁是一类。

**怎么打破维度诅咒？**
答：砍掉无意义的维度。这就是为什么我们在第四步画图时，使用了 PCA（主成分分析）。

<br><hr>

#### 第六步：【拓展提高】硬核数学时刻 —— 手搓 PCA 与 线性变换矩阵
<hr>

很多人在刚才调用 `PCA(n_components=2)` 时，觉得它就是一个压缩软件，塞进去 4 维，吐出来 2 维。

**如果你只学到这一层，你永远无法理解深度学习中的卷积神经网络（CNN）是怎么运作的。**
接下来，我们要彻底抛弃 `sklearn` 的调包，完全使用 `numpy`，从纯数学的角度，带你手工推导并搓出一个 PCA。

我们要解答的核心问题是：**矩阵（Matrix）到底是个什么东西？**

绝大多数人以为，矩阵就是一个装满了数字的 Excel 表格。
错。
**矩阵，是一个对空间进行扭曲、拉伸、旋转的“动作（算子）”。**

假设你手里有一个矩阵 $A$：
当你把一个数据向量 $X$ 乘上这个矩阵 $A$ 时（也就是执行 $X \cdot A$），你其实并不是在做简单的乘法，你是在对 $X$ 所在的那个宇宙执行了一次**“线性变换”**。
矩阵 $A$ 把原本横平竖直的空间坐标轴，硬生生地扯斜了、旋转了。

**PCA 在干什么？**
我们的球员数据原本在四维空间里。这个四维空间的坐标轴是乱指的，数据在里面看起来像一团没有规律的面团。
PCA 运用了高等代数中最核心的魔法——**特征值与特征向量分解（Eigen Decomposition）**，它在做一件事：

1. 寻找面团被拉伸得最长的那几个方向（方差最大的方向）。
2. 把这些方向抽取出来，组成一个新的矩阵。
3. 拿这个新矩阵去乘以原来的面团。
4. 面团瞬间被旋转，正正好好地对准了全新的坐标轴。我们只要保留最长的那两根轴（主成分），其余的直接丢掉，压缩就完成了。

下面，请看纯手工打造的 PCA 代码，领略矩阵变换的暴力美学：

In [ ]:
# 【纯手工打造 PCA】

# 1. 第一步：数据中心化（把整团数据平移，让它的中心点正好落在宇宙坐标系的原点 (0,0,0,0)）
mean_vector = np.mean(X_scaled, axis=0)
X_centered = X_scaled - mean_vector

# 2. 第二步：计算协方差矩阵（Covariance Matrix）
# 这是用来衡量这四个特征之间是协同变化的，还是互相抵触的。
cov_matrix = np.cov(X_centered, rowvar=False)

# 3. 第三步：极其关键的——特征值分解！
# 找出这个面团在各个方向上被拉扯的力道（特征值 eigenvalues）
# 以及拉扯的具体方向标杆（特征向量 eigenvectors）
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# 注意：np.linalg.eigh 算出来的特征值是从小到大排的。
# 我们需要的是被拉扯得最狠的方向，所以要把它们倒序排，让最大的排在最前面。
sorted_index = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_index]
eigenvectors = eigenvectors[:, sorted_index]

# 4. 第四步：构建我们的“空间旋转机器（变换矩阵）”
# 因为我们只想降到 2 维，所以我们只抽取排在最前面的 2 根柱子（最强特征向量）
transform_matrix = eigenvectors[:, :2]

print("我们纯手工算出来的空间变换矩阵长这样：")
print(transform_matrix)
print("看！它本质上就是一个 (4行 x 2列) 的数字阵列。")

# 5. 第五步：执行降维打击！
# 拿我们中心化后的原始数据 X_centered，去乘以这个变换矩阵。
# 这就是一个最纯粹的矩阵乘法（线性投影），把 4 维空间的数据生生砸到了这个 2 维的倾斜切面上。
X_pca_manual = np.dot(X_centered, transform_matrix)

In [ ]:
# 我们来对比一下，手工搓的 PCA，和刚才用 sklearn 调包的 PCA，算出来的结果一样吗？
print("\n【结果对比】")
print("Sklearn 调包结果的前两行数据：")
print(X_pca[:2])

print("\n纯手工矩阵变换结果的前两行数据：")
print(X_pca_manual[:2])

# 你会发现数字一模一样！（符号相反是正常的，因为在几何中，顺时针旋转 90 度和逆时针旋转 90 度的坐标镜像等价，不影响相对位置）

##### 为什么你要提前懂这个？（为 CNN 铺路）

很快，在后续的深度学习课程中，我们会学到处理图像的**卷积神经网络（CNN）**。

图像本质上也是矩阵。在 CNN 中，有一层叫做“卷积层”。
里面放着的东西叫做“卷积核（Kernel / Filter）”。

那个卷积核，和我们刚刚手工算出来的这个 `transform_matrix` 本质上是一模一样的东西！
它们全都是一个包含着特定数字的矩阵。
当图像乘上（卷积）这个矩阵时，并不是在做普通的乘法。图像在经历一次又一次的**空间变换**。
有的矩阵负责把图像里的横线拉长；有的矩阵负责把图像变模糊；有的矩阵负责抠出图像的轮廓。

所谓人工智能在“学习”，本质上就是机器在无数次失败和求导中，**疯狂地修改这个矩阵里装的数字**，直到这个矩阵能够完美地把猫和狗扭曲到不同的宇宙维度里，从而实现完美的分类。

所有的魔法，都建立在这个简单的 `np.dot(X, Matrix)` 之上。

#### 总结

| 概念 | 大白话解释 | 核心作用 / 注意事项 |
|------|------|------|
| **K-Means** | 一群没有标签的数据，自己抱团取暖找同类。 | 需要指定分几类（K）。 |
| **k-means++** | KMeans 的升级版启动器。 | 让最初始的几个带头大哥站得尽量远，避免模型陷入死循环瞎聚类。必用。 |
| **轮廓系数** | 聚类效果的裁判打分表（-1 到 1 分）。 | 用来辅助我们寻找那个完美的 K 值。 |
| **维度诅咒** | 当特征多到离谱时，所有东西的距离都变得一样远。 | 距离失效了，KNN 和 K-Means 就会彻底瘫痪。 |
| **矩阵变换** | 不是表格，而是扭曲、旋转空间的算子。 | 这是深度学习（尤其是 CNN）处理高维特征的最底层基石。 |

<br>

> **关键点**：今天，我们首次跨入了无监督学习的大门。从现在起，请把模型计算的过程，具象化为空间里的几何拉扯。不管是支持向量机修墙，还是 K-Means 找抱团点，亦或是 PCA 旋转视角，**机器学习的本质是在高维几何空间中玩弄数据点**。

<br>

<hr><hr>

## 实验七完成
<hr>

##### 此实验教材最近更新时间 2026年3月17日
<hr><hr>